In [ ]:
import sys

print(sys.version)

3.11.12 (main, Apr  9 2025, 08:55:54) [GCC 11.4.0]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

train_df = pd.read_excel('/content/drive/MyDrive/train_df_v3.xlsx')

In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50521 entries, 0 to 50520
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id              50521 non-null  object
 1   texts           50521 non-null  object
 2   labels          50521 non-null  int64 
 3   filtered_texts  50521 non-null  object
dtypes: int64(1), object(3)
memory usage: 1.5+ MB


In [ ]:
train_df.drop_duplicates(subset='filtered_texts', keep='first', inplace=True, ignore_index=True)

In [ ]:
train_df

,id,texts,labels,filtered_texts
0,skyn****,희플 고급야구 ㄱㄱ,0,희플 고급야구 ㄱㄱ
1,깔싸미,공민규는 몸도 안움직이네,0,공민규는 몸도 안움직이네
2,JAMES,구단 해체...감독 경질...,0,구단 해체...감독 경질...
3,2021039,투코 좋다,0,투코 좋다
4,hjan****,구자욱은 ... 어디감,0,구자욱은 ... 어디감
...,...,...,...,...
50397,꽃향기,후쿠시마대구는 여권발급 얼마나 걸려요?,3,후쿠시마대구는 여권발급 얼마나 걸려요?
50398,우성테크,북한에 주적 대구,3,북한에 주적 대구
50399,제우스,광주박멸,3,광주박멸
50400,하정명,전라도 OOO,3,전라도 OOO


In [ ]:
train_df['labels'].value_counts()

,count
labels,
0,48927
1,749
3,432
2,294


In [ ]:
# 0번 데이터 40퍼 개수 구하기
((749+432+294)/2)*6

4425.0

In [ ]:
train_0 = train_df[train_df['labels'] == 0] # 레이블 0만 추출
train_0_set = train_0.sample(n=4425, random_state = 0) # 레이블 0 데이터 프레임에서 랜덤으로 n개 추출
train_notclean = train_df[train_df['labels'] != 0] # labels = 0 인 값 제거한 데이터 프레임
train_df2 = pd.concat([train_0_set, train_notclean])

In [ ]:
train_df2

,id,texts,labels,filtered_texts
8275,KOREASTH,기아 야옹이즈,0,기아 야옹이즈
48477,assa****,밑으로 가면 누굴 패고 다녔길래 ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ...,0,밑으로 가면 누굴 패고 다녔길래 ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ...
20650,vi,허..,0,허..
43268,21c,우리 현종이도 저런 폭투를 한 적이 있지..,0,우리 현종이도 저런 폭투를 한 적이 있지..
44204,ryua****,삼성이 스스로 기아 도와 주네 ㅋㅋ,0,삼성이 스스로 기아 도와 주네 ㅋㅋ
...,...,...,...,...
50397,꽃향기,후쿠시마대구는 여권발급 얼마나 걸려요?,3,후쿠시마대구는 여권발급 얼마나 걸려요?
50398,우성테크,북한에 주적 대구,3,북한에 주적 대구
50399,제우스,광주박멸,3,광주박멸
50400,하정명,전라도 OOO,3,전라도 OOO


In [ ]:
train_df2.to_excel('kcbert_train_60per_v2.xlsx')

### 텍스트 전처리

In [ ]:
!pip3 install soynlp emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 416.8/416.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.4/431.4 kB 27.9 MB/s eta 0:00:00


In [ ]:
import re
import emoji
from soynlp.normalizer import repeat_normalize

pattern = re.compile(f'[^ .,?!/@$%~％·∼()\x00-\x7Fㄱ-ㅣ가-힣]+')
url_pattern = re.compile(
    r'https?:\/\/(www\.)?[-a-zA-Z0-9@:%._\+~#=]{1,256}\.[a-zA-Z0-9()]{1,6}\b([-a-zA-Z0-9()@:%_\+.~#?&//=]*)')

def clean(x):
    x = pattern.sub(' ', x)
    x = emoji.replace_emoji(x, replace='') #emoji 삭제
    x = url_pattern.sub('', x)
    x = x.strip()
    x = repeat_normalize(x, num_repeats=2)
    return x

# texts 컬럼에서 특수문자를 제거
train_df2['clean_texts'] = train_df2['filtered_texts'].apply(clean)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    train_df2['filtered_texts'], train_df2['labels'], test_size=0.3, random_state = 0
)

In [ ]:
X_train = X_train.tolist()
y_train = y_train.tolist()

X_val = X_val.tolist()
y_val = y_val.tolist()

"X_test = test_df['texts'].tolist()\n#X_test = test_df['processed_text'].tolist()\ny_test = test_df['labels'].tolist()"

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification


tokenizer = AutoTokenizer.from_pretrained("beomi/kcbert-base")
model = AutoModelForSequenceClassification.from_pretrained("beomi/kcbert-base", num_labels = 4)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/250k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at beomi/kcbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW

class CustomDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        text = self.texts[index]
        label = self.labels[index]

        encoding = self.tokenizer(text, padding='max_length', truncation=True, max_length=self.max_length, return_tensors='pt')
        input_ids = encoding['input_ids'].squeeze()
        attention_mask = encoding['attention_mask'].squeeze()

        return {'input_ids': input_ids, 'attention_mask': attention_mask, 'label': label}

In [ ]:
# 원하는 최대 시퀀스 길이
max_length = 64

t_labels = torch.tensor(y_train, dtype=torch.long)
v_labels = torch.tensor(y_val, dtype=torch.long)
train_dataset = CustomDataset(X_train, t_labels, tokenizer, max_length)
val_dataset = CustomDataset(X_val, v_labels , tokenizer, max_length)

### 하이퍼 파라미터 튜닝

In [ ]:
from transformers import get_linear_schedule_with_warmup

def train_and_evaluate(model, learning_rate, epochs, batch_size, device):
    # 데이터 로더 생성
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    valid_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)

    # 모델
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    # 옵티마이저 및 손실 함수 설정
    optimizer = AdamW(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()

    total_steps = len(train_dataloader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for batch in train_dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_dataloader)
        print(f"Epoch {epoch+1}/{epochs} - Avg Loss: {avg_loss:.4f}")

        model.eval()
        val_total_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():
            for val_batch in valid_dataloader:
                val_input_ids = val_batch['input_ids'].to(device)
                val_attention_mask = val_batch['attention_mask'].to(device)
                val_labels = val_batch['label'].to(device)

                # 손실 계산
                val_outputs = model(val_input_ids, attention_mask=val_attention_mask)
                val_logits = val_outputs.logits
                val_loss = criterion(val_logits, val_labels)
                val_total_loss += val_loss.item()

                # 정확도 계산
                val_preds = val_logits.argmax(dim=1)
                correct += (val_preds == val_labels).sum().item()
                total += val_labels.size(0)

        val_avg_loss = val_total_loss / len(valid_dataloader)
        val_accuracy = correct / total
        print(f"Validation Loss: {val_avg_loss:.4f} - Validation Accuracy: {val_accuracy:.4f}")

    return val_avg_loss, val_accuracy

# 하이퍼파라미터 설정
learning_rates = [2e-5, 3e-05]
epochs_list = [5, 6, 7, 8]
batch_sizes = [8, 16]

best_val_accuracy = 0
best_hyperparams = {}

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

for learning_rate in learning_rates:
    for epochs in epochs_list:
      for batch_size in batch_sizes:
        print(f"Training with learning rate: {learning_rate}, epochs: {epochs}, batch_size: {batch_size}")
        model = AutoModelForSequenceClassification.from_pretrained("beomi/kcbert-base", num_labels = 4)  # 모델을 다시 초기화
        val_loss, val_accuracy = train_and_evaluate(model, learning_rate, epochs, batch_size, device)

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_hyperparams = {'learning_rate': learning_rate, 'epochs': epochs, 'batch_size': batch_size}

print(f"Best Hyperparameters: {best_hyperparams} with Validation Accuracy: {best_val_accuracy:.4f}")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at beomi/kcbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training with learning rate: 2e-05, epochs: 5, batch_size: 8


/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1/5 - Avg Loss: 0.5534
Validation Loss: 0.3578 - Validation Accuracy: 0.8876
Epoch 2/5 - Avg Loss: 0.2543
Validation Loss: 0.3593 - Validation Accuracy: 0.8808
Epoch 3/5 - Avg Loss: 0.1020
Validation Loss: 0.4136 - Validation Accuracy: 0.8831
Epoch 4/5 - Avg Loss: 0.0356
Validation Loss: 0.4609 - Validation Accuracy: 0.8847
Epoch 5/5 - Avg Loss: 0.0177


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at beomi/kcbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Validation Loss: 0.5608 - Validation Accuracy: 0.8842
Training with learning rate: 3e-05, epochs: 7, batch_size: 16
Epoch 1/7 - Avg Loss: 0.5693
Validation Loss: 0.3831 - Validation Accuracy: 0.8712
Epoch 2/7 - Avg Loss: 0.2722
Validation Loss: 0.3479 - Validation Accuracy: 0.8859
Epoch 3/7 - Avg Loss: 0.1158
Validation Loss: 0.4580 - Validation Accuracy: 0.8446
Epoch 4/7 - Avg Loss: 0.0354
Validation Loss: 0.4730 - Validation Accuracy: 0.8831
Epoch 5/7 - Avg Loss: 0.0138
Validation Loss: 0.5108 - Validation Accuracy: 0.8881
Epoch 6/7 - Avg Loss: 0.0106
Validation Loss: 0.5190 - Validation Accuracy: 0.8932
Epoch 7/7 - Avg Loss: 0.0051


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at beomi/kcbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Validation Loss: 0.5295 - Validation Accuracy: 0.8955
Training with learning rate: 3e-05, epochs: 8, batch_size: 8
Epoch 1/8 - Avg Loss: 0.5427
Validation Loss: 0.3572 - Validation Accuracy: 0.8825
Epoch 2/8 - Avg Loss: 0.2583
Validation Loss: 0.4000 - Validation Accuracy: 0.8740
Epoch 3/8 - Avg Loss: 0.1012
Validation Loss: 0.5094 - Validation Accuracy: 0.8650
Epoch 4/8 - Avg Loss: 0.0397
Validation Loss: 0.4952 - Validation Accuracy: 0.8870
Epoch 5/8 - Avg Loss: 0.0145
Validation Loss: 0.5661 - Validation Accuracy: 0.8870
Epoch 6/8 - Avg Loss: 0.0051
Validation Loss: 0.5846 - Validation Accuracy: 0.8825
Epoch 7/8 - Avg Loss: 0.0047
Validation Loss: 0.5944 - Validation Accuracy: 0.8876
Epoch 8/8 - Avg Loss: 0.0028


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at beomi/kcbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Validation Loss: 0.6052 - Validation Accuracy: 0.8802
Training with learning rate: 3e-05, epochs: 8, batch_size: 16
Epoch 1/8 - Avg Loss: 0.5622
Validation Loss: 0.3850 - Validation Accuracy: 0.8723
Epoch 2/8 - Avg Loss: 0.2647
Validation Loss: 0.4192 - Validation Accuracy: 0.8588
Epoch 3/8 - Avg Loss: 0.1171
Validation Loss: 0.4561 - Validation Accuracy: 0.8661
Epoch 4/8 - Avg Loss: 0.0403
Validation Loss: 0.5351 - Validation Accuracy: 0.8712
Epoch 5/8 - Avg Loss: 0.0163
Validation Loss: 0.5583 - Validation Accuracy: 0.8757
Epoch 6/8 - Avg Loss: 0.0085
Validation Loss: 0.5713 - Validation Accuracy: 0.8780
Epoch 7/8 - Avg Loss: 0.0058
Validation Loss: 0.5801 - Validation Accuracy: 0.8797
Epoch 8/8 - Avg Loss: 0.0035
Validation Loss: 0.5808 - Validation Accuracy: 0.8802
Best Hyperparameters: {'learning_rate': 3e-05, 'epochs': 6, 'batch_size': 8} with Validation Accuracy: 0.8955


In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 하이퍼파라미터 설정
learning_rate = 3e-05
epochs = 6
batch_size = 8

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)

# 옵티마이저 및 손실 함수 설정
optimizer = AdamW(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

# 모델 재학습
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in train_dataloader:
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        labels = batch['label']

        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        # 그래디언트 초기화
        optimizer.zero_grad()
        # 모델에 입력을 주어 예측을 생성합니다.
        outputs = model(input_ids, attention_mask=attention_mask)
        # 모델 출력에서 로짓(분류에 대한 점수)을 얻습니다.
        logits = outputs.logits
        # 손실을 계산합니다.
        loss = criterion(logits, labels)
        # 역전파를 통해 그래디언트 계산
        loss.backward()
        # 옵티마이저를 사용해 가중치를 업데이트
        optimizer.step()
        # 에포크 전체 손실을 누적합니다.
        total_loss += loss.item()

    # 에포크 평균 손실 계산
    avg_loss = total_loss / len(train_dataloader)
    # 에포크별 손실 출력
    print(f"Epoch {epoch+1}/{epochs} - Avg Loss: {avg_loss:.4f}")

    # 모델 평가
    model.eval()
    val_total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for val_batch in valid_dataloader:
            # Validation 데이터 가져오기
            val_input_ids = val_batch['input_ids']
            val_attention_mask = val_batch['attention_mask']
            val_labels = val_batch['label']

            val_input_ids = val_input_ids.to(device)
            val_attention_mask = val_attention_mask.to(device)
            val_labels = val_labels.to(device)

            # 모델 예측
            val_outputs = model(val_input_ids, attention_mask=val_attention_mask)
            val_logits = val_outputs.logits

            # 손실 계산
            val_loss = criterion(val_logits, val_labels)
            val_total_loss += val_loss.item()

            # 정확도 계산
            val_preds = val_logits.argmax(dim=1)
            correct += (val_preds == val_labels).sum().item()
            total += val_labels.size(0)

    val_avg_loss = val_total_loss / len(valid_dataloader)
    val_accuracy = correct / total
    print(f"Validation Loss: {val_avg_loss:.4f} - Validation Accuracy: {val_accuracy:.4f}")


Epoch 1/6 - Avg Loss: 0.0460
Validation Loss: 0.6293 - Validation Accuracy: 0.8853
Epoch 2/6 - Avg Loss: 0.0199
Validation Loss: 0.5679 - Validation Accuracy: 0.8785
Epoch 3/6 - Avg Loss: 0.0219
Validation Loss: 0.6949 - Validation Accuracy: 0.8621
Epoch 4/6 - Avg Loss: 0.0392
Validation Loss: 0.6414 - Validation Accuracy: 0.8661
Epoch 5/6 - Avg Loss: 0.0462
Validation Loss: 0.6511 - Validation Accuracy: 0.8768
Epoch 6/6 - Avg Loss: 0.0372
Validation Loss: 0.7843 - Validation Accuracy: 0.8672


In [ ]:
import torch

model_save_path = '/content/drive/MyDrive/model/KcBERT_v7_60per.pth'
torch.save(model.state_dict(), model_save_path)